In [128]:
from glob import glob
import os
import pickle
import h5py
def load_h5(filename):
    data = {}

    with h5py.File(filename, 'r') as f:
        data['X_train'] = f['X_train'][:]
        data['y_train'] = f['y_train'][:]
        data['X_test'] = f['X_test'][:]
        data['y_test'] = f['y_test'][:]

        if 'y_pred_proba' in f:
            data['y_pred_proba'] = f['y_pred_proba'][:]
        else:
            data['y_pred_proba'] = None

        data['res'] = dict(f.attrs)

    return data

def pretty_print_rules(rules):
    for rule, prediction, support, log_odds in rules:
        s = ''
        for idx, thr, comp in rule:
            s += f'X[{idx}] {comp} {round(thr, 2)}\tAND\t'
        s = s[:-4] + f'=> {round(prediction, 2)} \t ({support}) ({log_odds})'
        print(s)

In [196]:
filename = 'res/titanic/ScoreCard3/43946797f2ae498ea1791b475565407c.csv'[:-4]+'.pkl'

with open(filename, 'rb') as f:
    s = pickle.load(f)

sorted([score for base, rule, supp, score in s.scorecard])[:10]+sorted([score for base, rule, supp, score in s.scorecard])[-10:]

[-13, -9, -7, -6, -4, -4, -3, -3, -3, -3, 6, 6, 7, 8, 8, 8, 8, 11, 14, 19]

In [197]:
X_train = load_h5(filename.replace('.pkl', '.h5').replace('ScoreCard3', 'RuleCard'))['X_train']
y_train = load_h5(filename.replace('.pkl', '.h5').replace('ScoreCard3', 'RuleCard'))['y_train']

X_train.shape

(712, 8)

In [198]:
s

,rules,"[([(3, ...), (6, ...), ...], ...), ([(3, ...), (6, ...), ...], ...), ...]"
,lr,0.3
,base_log_odds,-0.4915098344666992
,classes,"[0, 1]"
,PDO,20
,odds0,1
,score0,0


In [206]:
min_thr = 240
scorecard = []

for base, rules, supp, score in s.scorecard:
    if supp < min_thr:
        continue
    scorecard.append(
        (base, rules, supp, score)
    )

print(len(scorecard))

17


In [335]:
for i in range(X_train.shape[0]):
    activated_rules = []
    has_pos = False
    has_neg = False
    max_pos = -float('inf')
    max_neg = float('inf')
    for base, rules, supp, score in scorecard:
        if s._get_activation(rules=rules, X=X_train[i].reshape(1, -1)).sum() == 1:
            has_pos |= score > 0
            has_neg |= score < 0
            max_pos = max(max_pos, score)
            max_neg = min(max_neg, score)
            activated_rules.append((rules, score))
    if len(activated_rules) <= 4 and has_pos and has_neg:
        print(i, len(activated_rules), y_train[i], max_pos, max_neg, X_train[i, 3], sep='\t')

15	4	1	19	-1	0.0
51	4	0	1	-13	1.0
53	4	1	19	-1	0.0
62	4	1	19	-1	0.0
69	3	0	19	-6	0.0
79	4	1	19	-1	0.0
80	4	1	19	-1	0.0
81	4	1	19	-1	0.0
101	4	1	19	-6	0.0
122	3	1	19	-1	0.0
141	3	1	19	-1	0.0
143	3	1	19	-1	0.0
153	4	1	19	-1	0.0
166	4	1	19	-6	0.0
172	4	1	1	-13	1.0
187	3	1	19	-1	0.0
188	4	1	19	-1	0.0
201	4	0	1	-13	1.0
208	4	1	19	-1	0.0
209	4	1	19	-1	0.0
219	2	1	1	-13	1.0
255	4	1	19	-1	0.0
274	3	1	1	-13	1.0
293	4	1	19	-1	0.0
297	4	1	19	-1	0.0
301	4	1	19	-1	0.0
304	4	1	19	-1	0.0
309	3	1	19	-1	0.0
314	3	1	19	-1	0.0
358	4	1	19	-1	0.0
366	4	1	19	-1	0.0
368	4	1	19	-1	0.0
391	3	1	1	-13	1.0
409	2	1	1	-13	1.0
410	4	1	19	-1	0.0
413	4	1	19	-1	0.0
461	4	1	19	-1	0.0
468	4	1	19	-1	0.0
494	2	1	1	-13	1.0
541	4	1	19	-1	0.0
559	4	1	19	-1	0.0
561	4	1	1	-13	1.0
563	4	1	1	-13	1.0
564	4	1	19	-1	0.0
568	4	1	19	-1	0.0
579	4	1	1	-13	1.0
599	4	1	19	-6	0.0
615	4	1	19	-6	0.0
633	3	1	19	-1	0.0
643	3	1	19	-1	0.0
647	3	1	19	-1	0.0
664	4	1	19	-1	0.0
669	4	1	19	-1	0.0
674	4	1	1	-13	1.0
680	4	1	19	-1	0.0
686	3	1	19	-1	0.0


In [342]:
def pretty_print_rules(rules):
    for _, rule, support, prediction in rules:
        s = ''
        for idx, thr, comp in rule:
            s += f'X[{idx}] {comp} {round(thr, 2)}\tAND\t'
        s = s[:-4] + f'=> {round(prediction, 2)} \t ({support})'
        print(s)


activated_rules = []
for base, rules, supp, score in scorecard:
    if s._get_activation(rules=rules, X=X_train[101].reshape(1, -1)).sum() == 1:
        activated_rules.append((base, rules, supp, score))

pretty_print_rules(sorted(activated_rules, key=lambda x: x[-1]))

X[4] > 2.5	=> -6 	 (391)
X[2] <= 2.5	AND	X[4] > 2.5	=> -3 	 (382)
X[4] > 2.5	AND	X[5] > 24.0	AND	X[5] <= 49.5	=> -2 	 (382)
X[3] <= 0.5	=> 19 	 (253)


In [343]:
pretty_print_rules(sorted(scorecard, key=lambda x: x[-1]))

X[3] > 0.5	=> -13 	 (459)
X[3] > 0.5	AND	X[6] > 3.5	AND	X[6] <= 77.0	=> -9 	 (443)
X[4] > 2.5	=> -6 	 (391)
X[5] > 35.5	AND	X[5] <= 37.5	AND	X[3] > 0.5	=> -4 	 (382)
X[6] > 1.5	AND	X[3] > 0.5	AND	X[6] > 3.5	=> -4 	 (444)
X[6] > 1.5	AND	X[4] > 1.5	AND	X[6] > 6.5	=> -3 	 (503)
X[2] <= 2.5	AND	X[4] > 2.5	=> -3 	 (382)
X[4] > 2.5	AND	X[5] > 24.0	AND	X[5] <= 49.5	=> -2 	 (382)
X[3] > 0.5	AND	X[1] <= 2.5	AND	X[1] <= 0.5	=> -1 	 (351)
X[6] > 1.5	AND	X[7] <= 52.28	AND	X[6] > 6.5	=> -1 	 (554)
X[6] > 1.5	AND	X[5] > 30.5	AND	X[5] <= 37.5	=> -1 	 (560)
X[6] > 1.5	AND	X[2] <= 2.5	AND	X[6] > 5.5	=> -1 	 (610)
X[6] > 1.5	AND	X[6] > 6.5	AND	X[4] > 1.5	=> -1 	 (503)
X[5] > 30.5	AND	X[5] > 31.5	AND	X[4] <= 2.5	=> 1 	 (269)
X[6] > 1.5	AND	X[7] <= 52.28	AND	X[6] <= 36.25	=> 1 	 (459)
X[6] > 1.5	AND	X[3] <= 0.5	AND	X[6] > 3.5	=> 7 	 (244)
X[3] <= 0.5	=> 19 	 (253)


In [344]:
activated_rules2 = []
for base, rules, supp, score in scorecard:
    if s._get_activation(rules=rules, X=X_train[51].reshape(1, -1)).sum() == 1:
        activated_rules2.append((base, rules, supp, score))

pretty_print_rules(sorted(activated_rules2, key=lambda x: x[-1]))

X[3] > 0.5	=> -13 	 (459)
X[3] > 0.5	AND	X[6] > 3.5	AND	X[6] <= 77.0	=> -9 	 (443)
X[6] > 1.5	AND	X[3] > 0.5	AND	X[6] > 3.5	=> -4 	 (444)
X[5] > 30.5	AND	X[5] > 31.5	AND	X[4] <= 2.5	=> 1 	 (269)


In [345]:
import pandas as pd

col_names = {
    0: 'Parch',
    1: 'SibSp',
    2: 'Cabin_letter',
    3: 'Sex',
    4: 'Pclass',
    5: 'Cabin_n',
    6: 'Age',
    7: 'Fare',
}

def simplify_rule(rule, r=3):
    bounds = {}

    for idx, thr, comp in rule:
        if idx not in bounds:
            bounds[idx] = {"low": None, "high": None}

        if comp in (">", ">="):
            bounds[idx]["low"] = thr
        elif comp in ("<", "<="):
            bounds[idx]["high"] = thr

    parts = []
    for idx, b in bounds.items():
        low, high = b["low"], b["high"]

        if low is not None and high is not None:
            parts.append(f"{round(low, 3)} < {col_names[idx]} <= {round(high, 3)}")
        elif low is not None:
            parts.append(f"{round(low, 3)} < {col_names[idx]}")
        elif high is not None:
            parts.append(f"{col_names[idx]} <= {round(high, 3)}")

    return " & ".join(parts)

def rules_table(rules):
    rows = []

    for _, rule, support, prediction in rules:
        cond = simplify_rule(rule)
        score = prediction * support
        rows.append((cond, prediction))

    df = pd.DataFrame(rows, columns=["rule", "score"])
    #df = df.reindex(df.score.abs().sort_values(ascending=False).index)
    
    return df

rules_table(sorted(scorecard, key=lambda x: -x[-1])).style.background_gradient(
    subset=["score"],
    cmap="bwr",
    vmin=-rules_table(scorecard)["score"].abs().max(),
    vmax=rules_table(scorecard)["score"].abs().max()
)

,rule,score
0,Sex <= 0.5,19
1,3.5 < Age & Sex <= 0.5,7
2,31.5 < Cabin_n & Pclass <= 2.5,1
3,1.5 < Age <= 36.25 & Fare <= 52.277,1
4,0.5 < Sex & SibSp <= 0.5,-1
5,6.5 < Age & Fare <= 52.277,-1
6,1.5 < Age & 30.5 < Cabin_n <= 37.5,-1
7,5.5 < Age & Cabin_letter <= 2.5,-1
8,6.5 < Age & 1.5 < Pclass,-1
9,2.5 < Pclass & 24.0 < Cabin_n <= 49.5,-2


In [346]:
rules_table(sorted(activated_rules, key=lambda x: -x[-1])).style.background_gradient(
    subset=["score"],
    cmap="bwr",
    vmin=-rules_table(scorecard)["score"].abs().max(),
    vmax=rules_table(scorecard)["score"].abs().max()
)

,rule,score
0,Sex <= 0.5,19
1,2.5 < Pclass & 24.0 < Cabin_n <= 49.5,-2
2,Cabin_letter <= 2.5 & 2.5 < Pclass,-3
3,2.5 < Pclass,-6


In [347]:
rules_table(sorted(activated_rules2, key=lambda x: -x[-1])).style.background_gradient(
    subset=["score"],
    cmap="bwr",
    vmin=-rules_table(scorecard)["score"].abs().max(),
    vmax=rules_table(scorecard)["score"].abs().max()
)

,rule,score
0,31.5 < Cabin_n & Pclass <= 2.5,1
1,3.5 < Age & 0.5 < Sex,-4
2,0.5 < Sex & 3.5 < Age <= 77.0,-9
3,0.5 < Sex,-13


In [348]:
def read_titanic(encode=True):
    target_col = 'Survived'
    df = pd.read_csv('../datasets/CLF/titanic.csv').drop(columns=['Embarked', 'PassengerId'])

    if encode:
        df.Sex = LabelEncoder().fit_transform(df.Sex)
        df.Cabin_n = LabelEncoder().fit_transform(df.Cabin_n)
        df.Cabin_letter = df.Cabin_letter.apply(lambda x: ord(x) - ord('A')) #lower = front

        #ct = ColumnTransformer([("cat", OneHotEncoder(), make_column_selector(dtype_include="object"))],
        #                       remainder='passthrough', verbose_feature_names_out=False, sparse_threshold=0, n_jobs=12)

        #df = pd.DataFrame(ct.fit_transform(df), columns=ct.get_feature_names_out())

    columns = set(df.columns.tolist()) - {target_col}

    return 'titanic', df[list(columns) + [target_col]].rename(columns={target_col: 'y'})

read_titanic(encode=False)[1]

,Age,Cabin_n,Fare,Cabin_letter,SibSp,Pclass,Sex,Parch,y
0,22.000000,33,7.2500,C,1,3,male,0,0
1,38.000000,85,71.2833,C,1,1,female,0,1
2,26.000000,33,7.9250,C,0,3,female,0,1
3,35.000000,123,53.1000,C,1,1,female,0,1
4,35.000000,33,8.0500,C,0,3,male,0,0
...,...,...,...,...,...,...,...,...,...
886,27.000000,33,13.0000,C,0,2,male,0,0
887,19.000000,42,30.0000,B,0,1,female,0,1
888,29.699118,33,23.4500,C,1,3,female,2,0
889,26.000000,148,30.0000,C,0,1,male,0,1


In [349]:
read_titanic(encode=False)[1].Cabin_letter.unique()

array(['C', 'E', 'G', 'D', 'A', 'B', 'F', 'T'], dtype=object)

In [350]:
import dataframe_image as dfi

In [351]:
rules_table(sorted(scorecard, key=lambda x: -x[-1])).to_excel('scorecard_global.xlsx')
rules_table(sorted(activated_rules, key=lambda x: -x[-1])).to_excel('scorecard_local.xlsx')
rules_table(sorted(activated_rules2, key=lambda x: -x[-1])).to_excel('scorecard2_local.xlsx')

In [387]:
tmp = pd.read_excel('scorecard_global2.xlsx', index_col=0)

def split_rule(s):
    splitted = s.replace('==', '=').replace('<=', '≤').replace('letter', 'let').replace('_letter', '_let').replace('female', 'F').replace('male', 'M').split(' & ')
    if len(splitted) == 1:
        return '', splitted[0]
    return sorted(splitted, key=len)

tmp[['','rule']] = tmp['rule'].apply(lambda s: pd.Series(split_rule(s)))
tmp[''] = tmp[''].apply(lambda x: f"{x}\xa0\xa0\xa0\xa0&" if len(x) > 2 else x)


styled = tmp[['', 'rule', 'score']].style.hide(axis="index").background_gradient(
    subset=["score"],
    cmap="bwr",
    vmin=-rules_table(scorecard)["score"].abs().max(),
    vmax=rules_table(scorecard)["score"].abs().max(),
).set_table_styles([{'selector': 'td', 'props': [('padding', '4px 6px')]}])

dfi.export(styled, "scorecard_global.png", table_conversion="matplotlib", dpi=600, )

styled

,rule,score
,Sex = 'F',19
3.5 < Age &,Sex = 'F',7
Pclass ≤ 2.5 &,31.5 < Cabin_n,1
Fare ≤ 52.3 &,1.5 < Age ≤ 36.3,1
Sex = 'M' &,SibSp ≤ 0.5,-1
6.5 < Age &,Fare ≤ 52.3,-1
1.5 < Age &,30.5 < Cabin_n ≤ 37.5,-1
5.5 < Age &,"Cabin_let is ['A', 'B']",-1
6.5 < Age &,1.5 < Pclass,-1
2.5 < Pclass &,24.0 < Cabin_n ≤ 49.5,-2


In [388]:
tmp = pd.read_excel('scorecard_local2.xlsx', index_col=0)

tmp[['','rule']] = tmp['rule'].apply(lambda s: pd.Series(split_rule(s)))
tmp[''] = tmp[''].apply(lambda x: f"{x}\xa0\xa0\xa0\xa0&" if len(x) > 2 else x)


styled = tmp[['', 'rule', 'score']].style.hide(axis="index").background_gradient(
    subset=["score"],
    cmap="bwr",
    vmin=-rules_table(scorecard)["score"].abs().max(),
    vmax=rules_table(scorecard)["score"].abs().max()
).set_table_styles([{'selector': 'td', 'props': [('padding', '4px 6px')]}])

dfi.export(styled, "scorecard_local.png", table_conversion="matplotlib", dpi=600)

styled

,rule,score
,Sex = 'F',19
2.5 < Pclass &,24.0 < Cabin_n ≤ 49.5,-2
2.5 < Pclass &,"Cabin_let is ['A', 'B']",-3
,2.5 < Pclass,-6


In [389]:
tmp = pd.read_excel('scorecard2_local2.xlsx', index_col=0)

tmp[['','rule']] = tmp['rule'].apply(lambda s: pd.Series(split_rule(s)))
tmp[''] = tmp[''].apply(lambda x: f"{x}\xa0\xa0\xa0\xa0&" if len(x) > 2 else x)


styled = tmp[['', 'rule', 'score']].style.hide(axis="index").background_gradient(
    subset=["score"],
    cmap="bwr",
    vmin=-rules_table(scorecard)["score"].abs().max(),
    vmax=rules_table(scorecard)["score"].abs().max()
).set_table_styles([{'selector': 'td', 'props': [('padding', '4px 6px')]}])

dfi.export(styled, "scorecard_local2.png", table_conversion="matplotlib", dpi=600)

styled

,rule,score
Pclass ≤ 2.5 &,31.5 < Cabin_n,1
3.5 < Age &,Sex = 'M',-4
Sex = 'M' &,3.5 < Age ≤ 77.0,-9
,Sex = 'M',-13


In [392]:
#top

print(real_y_train[101:102])

tmp = pd.DataFrame(real_X_train, columns=[x.replace('_letter', '_let') for x in df.columns][:-1]).map(lambda x: f'{round(x, 1)}'.replace('.0', '')).infer_objects().iloc[101:102]

tmp.Sex = tmp.Sex.apply(lambda x: "F" if x == '0' else "M")
tmp.Cabin_let = tmp.Cabin_let.apply(lambda x: chr(int(x)+ord('A')-1))

styled = tmp.style.hide(axis="index")\
    .set_table_styles([
              {"selector": "th",
               "props": [("writing-mode", "vertical-rl"),  # intestazioni verticali
                         ("text-orientation", "mixed"),  ('text-align', 'right')]},
                {'selector': 'td', 'props': [('text-align', 'center'), ('padding', '4px 11px')]}
          ])

#dfi.export(styled, "el_scorecard_local.png", table_conversion="matplotlib", dpi=600)

styled

[1.]


Age,Cabin_n,Fare,Cabin_let,SibSp,Pclass,Sex,Parch
1,37,15.7,B,0,3,F,2


In [391]:
#bottom

print(real_y_train[51:51+1])

tmp = pd.DataFrame(real_X_train, columns=[x.replace('_letter', '_let') for x in df.columns][:-1]).map(lambda x: f'{round(x, 1)}'.replace('.0', '')).infer_objects().iloc[51:51+1]

tmp.Sex = tmp.Sex.apply(lambda x: "F" if x == '0' else "M")
tmp.Cabin_let = tmp.Cabin_let.apply(lambda x: chr(int(x)+ord('A')))

tmp.style.hide(axis="index")\
    .set_table_styles([
              {"selector": "th",
               "props": [("writing-mode", "vertical-rl"),  # intestazioni verticali
                         ("text-orientation", "mixed"),  ('text-align', 'right')]},
                {'selector': 'td', 'props': [('text-align', 'center'), ('padding', '4px 11px')]}
          ])

[0.]


Age,Cabin_n,Fare,Cabin_let,SibSp,Pclass,Sex,Parch
50,48,55.9,E,1,1,M,0


In [379]:
read_titanic(encode=False)[1].Cabin_letter.unique()

array(['C', 'E', 'G', 'D', 'A', 'B', 'F', 'T'], dtype=object)

In [299]:
from readers import *
from sklearn.model_selection import train_test_split

df = read_titanic()[1]
X=df.iloc[:, :-1].values
y=df.iloc[:, -1].values

real_X_train, _, real_y_train, _ = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)



In [226]:
pd.DataFrame(X_train).head(6)

,0,1,2,3,4,5,6,7
0,0.0,0.0,2.0,1.0,3.0,37.0,29.699118,56.4958
1,0.0,0.0,2.0,1.0,2.0,37.0,29.699118,0.0000
2,0.0,0.0,2.0,1.0,1.0,89.0,29.699118,221.7792
3,1.0,0.0,2.0,0.0,3.0,37.0,18.000000,9.3500
4,1.0,1.0,2.0,0.0,2.0,37.0,31.000000,26.2500
5,0.0,0.0,2.0,1.0,3.0,37.0,21.000000,8.4333


In [227]:
pd.DataFrame(real_X_train, columns=list(df.columns)[:-1]).head(6)

,Age,Cabin_n,Fare,Cabin_letter,SibSp,Pclass,Sex,Parch
0,29.699118,37.0,56.4958,2.0,0.0,3.0,1.0,0.0
1,29.699118,37.0,0.0000,2.0,0.0,2.0,1.0,0.0
2,29.699118,89.0,221.7792,2.0,0.0,1.0,1.0,0.0
3,18.000000,37.0,9.3500,2.0,0.0,3.0,0.0,1.0
4,31.000000,37.0,26.2500,2.0,1.0,2.0,0.0,1.0
5,21.000000,37.0,8.4333,2.0,0.0,3.0,1.0,0.0


In [331]:
pd.DataFrame(real_X_train, columns=list(df.columns)[:-1]).sum()

Age             21208.199118
Cabin_n         27684.000000
Fare            22655.716300
Cabin_letter     1512.000000
SibSp             351.000000
Pclass           1644.000000
Sex               459.000000
Parch             278.000000
dtype: float64

In [333]:
pd.DataFrame(X_train).sum()

0      278.000000
1      351.000000
2     1512.000000
3      459.000000
4     1644.000000
5    27684.000000
6    21208.199118
7    22655.716300
dtype: float64

In [ ]:
col_names = {
    0: 'Parch',
    1: 'SibSp',
    2: 'Cabin_letter',
    3: 'Sex',
    4: 'Pclass',
    5: 'Cabin_n',
    6: 'Age',
    7: 'Fare',
}

In [168]:
df.is_recid

0       0.0
1       1.0
2       1.0
3       0.0
4       0.0
       ... 
7209    0.0
7210    0.0
7211    0.0
7212    0.0
7213    1.0
Name: is_recid, Length: 7214, dtype: float64